# MongoDB Atlas - push dataset to DB

Reads `data.csv.gz` (gzipped, pandas decompresses by extension) and pushes it to a MongoDB Atlas collection.

**Before running:** set the connection string in a terminal, then restart VS Code so the kernel inherits it.

```powershell
$env:MONGODB_URL = "mongodb+srv://<user>:<password>@<cluster>/?retryWrites=true&w=majority"
```

In [ ]:
import os
import sys

import certifi
import pandas as pd
import pymongo

print("python  :", sys.executable)
print("pandas  :", pd.__version__)
print("pymongo :", pymongo.__version__)

In [ ]:
DATABASE_NAME = "Proj1"
COLLECTION_NAME = "Proj1-Data"
DATA_FILE = "data.csv.gz"

MONGODB_URL = os.getenv("MONGODB_URL")
if not MONGODB_URL:
    raise EnvironmentError(
        "MONGODB_URL is not set. Set it in a terminal and restart VS Code:\n"
        '  $env:MONGODB_URL = "mongodb+srv://..."'
    )

print("MONGODB_URL loaded, host:", MONGODB_URL.split("@")[-1].split("/")[0])

In [ ]:
client = pymongo.MongoClient(MONGODB_URL, tlsCAFile=certifi.where())
client.admin.command("ping")
print("connection OK")

db = client[DATABASE_NAME]
collection = db[COLLECTION_NAME]

In [ ]:
df = pd.read_csv(DATA_FILE)
print("shape:", df.shape)
df.head()

In [ ]:
# 'id' is a row counter, not a feature - drop it unless schema.yaml expects it
df = df.drop(columns=["id"], errors="ignore").reset_index(drop=True)

records = df.to_dict(orient="records")
print("documents to insert:", len(records))
records[0]

In [ ]:
existing = collection.count_documents({})
print("documents already in collection:", existing)

# rerunning the insert cell on a non-empty collection duplicates data.
# uncomment to wipe first:
# collection.delete_many({})

In [ ]:
BATCH_SIZE = 10_000

inserted = 0
for start in range(0, len(records), BATCH_SIZE):
    batch = records[start : start + BATCH_SIZE]
    result = collection.insert_many(batch, ordered=False)
    inserted += len(result.inserted_ids)
    print(f"inserted {inserted}/{len(records)}")

print("done, total inserted:", inserted)

In [ ]:
print("count in collection:", collection.count_documents({}))

fetched = pd.DataFrame(list(collection.find().limit(5)))
fetched

In [ ]:
client.close()